# Bài 17/25 · LeetCode #101 — Symmetric Tree

**Mức độ:** Easy · **Chủ đề:** Binary Tree · Mirror Recursion · **Python:** 3  
**Đề gốc:** [https://leetcode.com/problems/symmetric-tree/](https://leetcode.com/problems/symmetric-tree/)

> Mục tiêu: hiểu cách tự suy luận từ ví dụ → viết code → trace trạng thái → chứng minh đúng → tự làm bài biến thể. Đọc Markdown, dự đoán rồi mới chạy cell code tiếp theo.

**Lưu ý:** đây là notebook dành cho người mới. Các cell TODO chỉ định nghĩa hàm, không được tự gọi khi chưa giải.

**Lưu ý học:** các ô TODO không được tự chạy; hãy tự làm, mở gợi ý từng mức rồi mới xem đáp án ở cuối.


## 1. Đọc đề và xác định đúng đầu vào/đầu ra

Trả `True` nếu cây nhị phân đối xứng qua trục đi qua node gốc. Nhánh **trái của bên trái** phải khớp với nhánh **phải của bên phải**.

### Ví dụ quan trọng

`[1,2,2,3,4,4,3] → True`; `[1,2,2,None,3,None,3] → False`; cây rỗng hoặc 1 node đều đối xứng.

Hãy tự trả lời: **đầu ra là số, boolean, danh sách, hay node?** Với bài cấu trúc dữ liệu, phải chú ý **node/tham chiếu** khác **giá trị số**.

## 2. Thiết lập môi trường để chạy notebook

Trong Jupyter, hãy chạy cell tạo lớp dữ liệu và hàm hỗ trợ trước các cell sau. **Lưu ý:** LeetCode có sẵn một số lớp hoặc API; khi submit chỉ sao chép riêng cell `class Solution`/lớp thiết kế, không sao chép hàm giả lập dùng cho notebook.

In [1]:
from __future__ import annotations
from typing import Optional
from collections import deque

class TreeNode:
    def __init__(self, val: int = 0, left=None, right=None):
        self.val, self.left, self.right = val, left, right
    def __repr__(self):
        return f"TreeNode({self.val})"

def tree(values):
    """Tạo cây từ danh sách level-order; None = ô trống."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0]); queue=deque([root]); pos=1
    while queue and pos < len(values):
        parent=queue.popleft()
        if pos < len(values):
            v=values[pos]; pos+=1
            if v is not None:
                parent.left=TreeNode(v); queue.append(parent.left)
        if pos < len(values):
            v=values[pos]; pos+=1
            if v is not None:
                parent.right=TreeNode(v); queue.append(parent.right)
    return root

def show_tree(root):
    """Đọc cấu trúc cây bằng BFS, không in quá nhiều None cuối."""
    if root is None:
        return []
    q=deque([root]); result=[]
    while q:
        p=q.popleft()
        if p is None:
            result.append(None)
        else:
            result.append(p.val); q.append(p.left); q.append(p.right)
    while result and result[-1] is None:
        result.pop()
    return result

### 2.1. `TreeNode` thật sự là gì? Học từ một nút đơn

**Binary Tree (cây nhị phân)** gồm các nút (*node*). Mỗi nút lưu đúng ba thông tin:
`val` (giá trị), `left` (tham chiếu đến con trái), `right` (tham chiếu đến con phải).

Quan trọng: `left`/`right` **không phải** là vị trí trong mảng; đó là liên kết đến một **đối tượng** `TreeNode` khác.
`None` nghĩa là *không có nút*. Một cây rỗng được biểu diễn bởi `root = None`.

Ví dụ cây `1` có con trái `2` và con phải `3`:

```text
      1
     / \
    2   3
```

Nhìn vào code dựng cây dưới đây, hãy dự đoán từng phép in trước khi chạy.

In [2]:
root_demo = TreeNode(1)
root_demo.left = TreeNode(2)
root_demo.right = TreeNode(3)
print("Gốc:", root_demo.val)
print("Con trái:", root_demo.left.val)
print("Con phải:", root_demo.right.val)
print("Con trái của nút 2:", root_demo.left.left)  # None

Gốc: 1
Con trái: 2
Con phải: 3
Con trái của nút 2: None


### 2.2. Tham chiếu (*reference*) và giá trị khác nhau thế nào?

`a = root_demo.left` **không sao chép** một nút mới: `a` và `root_demo.left` là hai tên cùng chỉ đến một đối tượng.
Vì vậy `a.val = 99` sẽ thay đổi giá trị trong cây gốc.

- `a is b`: hỏi **có cùng một đối tượng** hay không.
- `a.val == b.val`: hỏi **hai giá trị bằng nhau** hay không.
- Hai nút khác nhau có thể cùng giá trị; với so sánh cây, ta cần xét cả cấu trúc, không chỉ xét một con số.

Phần này đặc biệt quan trọng khi gắn nút con trong giải pháp chính.

In [3]:
x_demo = TreeNode(7)
y_demo = TreeNode(7)
z_demo = x_demo
print("Giá trị bằng nhau:", x_demo.val == y_demo.val)  # True
print("Là một đối tượng:", x_demo is y_demo)        # False
print("Chỉ đến cùng nút:", x_demo is z_demo)       # True
z_demo.val = 8
print("Giá trị của x sau khi đổi qua z:", x_demo.val)

Giá trị bằng nhau: True
Là một đối tượng: False
Chỉ đến cùng nút: True
Giá trị của x sau khi đổi qua z: 8


### 2.3. `None`, `is None` và thứ tự kiểm tra an toàn

Với `node = None`, viết `node.val` sẽ gây `AttributeError`: không thể truy cập `val` của vật không tồn tại.
Cách an toàn: kiểm tra `node is None` **trước**, rồi mới dùng `node.val`.

Trong Python, biểu thức `A and B` có **short-circuit** (đoản mạch): nếu `A` là `False` thì Python **không đánh giá** `B` nữa.
Tương tự, với `A or B`, nếu `A` là `True` thì không cần tính `B`.
Đây là lý do ta có thể xử lý trường hợp rỗng trước khi truy cập thuộc tính và tránh nhánh đệ quy vô ích.

In [4]:
nothing = None
print("Là nút rỗng?", nothing is None)
print("Không phải nút rỗng?", nothing is not None)
print("Đọc an toàn:", nothing.val if nothing is not None else "không có nút")
leaf_demo = TreeNode(42)
print("Lá có hai con rỗng:", leaf_demo.left is None and leaf_demo.right is None)

Là nút rỗng? True
Không phải nút rỗng? False
Đọc an toàn: không có nút
Lá có hai con rỗng: True


### 2.4. Hiểu danh sách level-order và vì sao phải giữ `None`

LeetCode thường hiển thị cây theo thứ tự từng tầng, trái trước phải sau (*level-order*).
Trong notebook, hàm `tree(values)` dựng cây từ dạng ấy để dễ thử, còn `show_tree(root)` đưa cây về danh sách để đọc.
**Đừng coi list đó là định nghĩa của cây**: thuật toán nộp LeetCode nhận `TreeNode`, không nhận danh sách.

Ví dụ `[1,2,None,3]` là: gốc 1 có con trái 2, nút 2 có con trái 3.
Nó khác `[1,None,2,3]`: gốc 1 có con phải 2, nút 2 có con trái 3.
Dù phần tử `3` xuất hiện cuối cả hai danh sách, vị trí con trỏ khác nhau hoàn toàn.

In [5]:
shape_a = tree([1, 2, None, 3])
shape_b = tree([1, None, 2, 3])
print("A:", show_tree(shape_a), "gốc trái:", shape_a.left)
print("B:", show_tree(shape_b), "gốc trái:", shape_b.left)
print("A.gốc.phải:", shape_a.right, "B.gốc.phải:", shape_b.right)
assert show_tree(shape_a) != show_tree(shape_b)

A: [1, 2, None, 3] gốc trái: TreeNode(2)
B: [1, None, 2, 3] gốc trái: None
A.gốc.phải: None B.gốc.phải: TreeNode(2)


### 2.5. Đệ quy (*recursion*) từ con số 0: hàm gọi lại chính nó

Đệ quy có hai mảnh **bắt buộc**:

1. **Base case** (trường hợp dừng): đơn giản đến mức trả lời ngay, không gọi tiếp.
2. **Recursive step** (bước rút gọn): gọi lại hàm trên bài toán nhỏ hơn, rồi dùng kết quả ấy.

**Call stack (ngăn xếp lời gọi)** giữ lại vị trí mà mỗi lần gọi đang chờ `return`.
Ví dụ `countdown(3)` cần đi `3 → 2 → 1 → 0` rồi quay trở ra.
Trên cây, bước nhỏ hơn tự nhiên là cây con `node.left` hoặc `node.right`.

Kí hiệu chiều cao `h`: một đường từ gốc xuống lá dài tối đa `h` nút, nên tầng gọi lồng nhau có thể đến `h`.
Đệ quy không tự biến mất bộ nhớ: mỗi lần gọi đang chờ chiếm một phần call stack.

In [6]:
def countdown_demo(n, level=0):
    print("  " * level + f"vào n={n}")
    if n == 0:
        print("  " * level + "đã đến base case")
        return 0
    result = 1 + countdown_demo(n - 1, level + 1)
    print("  " * level + f"trở lại n={n}, trả {result}")
    return result

assert countdown_demo(3) == 3

vào n=3
  vào n=2
    vào n=1
      vào n=0
      đã đến base case
    trở lại n=1, trả 1
  trở lại n=2, trả 2
trở lại n=3, trả 3


### 2.6. `Optional[TreeNode]` và `return` là gì?

`Optional[TreeNode]` (trong `typing`) mô tả kiểu dữ liệu có thể là một `TreeNode` **hoặc** `None`.
Đây là **type hint** (gợi ý kiểu), không làm Python tự kiểm tra cây hợp lệ trong lúc chạy.

`return` *trả kết quả về nơi gọi và kết thúc lượt chạy hiện tại*. `print` chỉ đưa thông tin lên màn hình;
một hàm chỉ `print(True)` mà không `return True` sẽ trả về `None` (không đúng kiểu `bool` mà nhiều bài đòi).

Vòng lặp `for`, phép `max`, phép `abs`, danh sách `[]`, biểu thức `x is None`, phép truy cập `.left` / `.right`
đều sẽ được giải thích lại ngay tại chỗ cần dùng ở phần sau.

In [7]:
def check_node_demo(node: Optional[TreeNode]) -> bool:
    if node is None:
        return False
    return node.val > 0

print("Kiểm tra None:", check_node_demo(None))
print("Kiểm tra nút 5:", check_node_demo(TreeNode(5)))
print("Kiểm tra nút -2:", check_node_demo(TreeNode(-2)))

Kiểm tra None: False
Kiểm tra nút 5: True
Kiểm tra nút -2: False


## 3. Python nền tảng — công cụ cần hiểu trước khi giải

Cách truy cập `node.left`, `node.right` và **đối chiếu chéo**. Ghi nhớ: mirror không phải chỉ so sánh dãy giá trị.

**Bài thí nghiệm:** thử thay dữ liệu trong cell dưới; các lệnh `print()` dùng để thấy giá trị trung gian, không phải mã submit.

In [8]:
root=tree([1,2,2,3,4,4,3]);print(show_tree(root))
print('Hai con:',root.left.val,root.right.val)
print('Đối xứng ngoài:',root.left.left.val,root.right.right.val)
print('Đối xứng trong:',root.left.right.val,root.right.left.val)

[1, 2, 2, 3, 4, 4, 3]
Hai con: 2 2
Đối xứng ngoài: 3 3
Đối xứng trong: 4 4


### 3.1. Đối xứng KHÔNG phải là “con trái bằng con phải”

Hình dung tấm gương đặt **dọc qua gốc**.

```text
         1              Trái ngoài 3 ↔ Phải ngoài 3
       /   \            Trái trong 4 ↔ Phải trong 4
      2     2
     / \   / \
    3   4 4   3
```

Hai con của gốc đều bằng `2` là **cần nhưng không đủ**. Toàn bộ cấu trúc còn phải phản chiếu.
Nếu phía trái có nút ở `left`, phía phải phải có nút tương ứng ở `right` (không phải ở `left`).

Tập giá trị có thể giống nhau dù một bên bị xếp sai vị trí. Việc đảo ngược thứ tự là **mấu chốt**.

In [9]:
correct_mirror = tree([1, 2, 2, 3, 4, 4, 3])
wrong_mirror = tree([1, 2, 2, 3, 4, 3, 4])
print("Hai cây có chung 7 giá trị nhưng cấu trúc đối xứng khác:")
print("Đúng:", show_tree(correct_mirror))
print("Sai:", show_tree(wrong_mirror))
print("Bốn nút cháu bên trái/phải:",
      correct_mirror.left.left.val, correct_mirror.right.right.val,
      correct_mirror.left.right.val, correct_mirror.right.left.val)

Hai cây có chung 7 giá trị nhưng cấu trúc đối xứng khác:
Đúng: [1, 2, 2, 3, 4, 4, 3]
Sai: [1, 2, 2, 3, 4, 3, 4]
Bốn nút cháu bên trái/phải: 3 3 4 4


### Tự kiểm tra ý nghĩa từng dòng

- Đầu vào của mỗi biểu thức là gì?
- Dòng nào tạo cấu trúc mới và dòng nào thay đổi cấu trúc hiện có?
- `True`/`False` và `None` có nghĩa gì trong trường hợp này?
- Nếu dữ liệu rỗng hoặc chỉ có một phần tử, đoạn mã có còn chạy được không?

## 4. Cách nghĩ trực giác (baseline)

Dễ nghĩ: lấy ảnh gương của cây rồi so sánh với cây ban đầu (đã biết kỹ thuật Same Tree). Tuy đúng nhưng ta tạo thêm nhiều node mới.

**Hãy dự đoán** kết quả của hàm baseline trên các ví dụ trước khi chạy.

In [10]:
def baseline(root):
    def mirror(node):
        if node is None: return None
        return TreeNode(node.val,mirror(node.right),mirror(node.left))
    def equal(a,b):
        if a is None or b is None: return a is b
        return a.val==b.val and equal(a.left,b.left) and equal(a.right,b.right)
    return equal(root,mirror(root))

In [11]:
for a in [[1,2,2,3,4,4,3],[1,2,2,None,3,None,3],[],[7]]:
    print(a,'->',baseline(tree(a)))

[1, 2, 2, 3, 4, 4, 3] -> True
[1, 2, 2, None, 3, None, 3] -> False
[] -> True
[7] -> True


### 4.1. Vì sao cách tạo cây phản chiếu (baseline) đúng nhưng chưa tối ưu?

Cách đơn giản dễ nghĩ: **sao chép cây rồi đảo con trái/con phải ở mọi nút**.
Sau đó so bản đảo với cây ban đầu bằng quy tắc Same Tree đã học.

`mirror(node)` tạo `TreeNode(node.val, mirror(node.right), mirror(node.left))`:
- Con **trái** của bản đảo lấy từ con **phải** của bản gốc.
- Con **phải** của bản đảo lấy từ con **trái** của bản gốc.

Cách này đúng về mặt tư duy, nhưng cần một cây sao chép `O(n)` nút.
Ta có thể bỏ cây trung gian: **trực tiếp đối chiếu các cặp nút ảnh gương** và trả `False` khi có khác biệt.
Phần tối ưu không chỉ là chạy nhanh hơn; quan trọng là không tạo thêm toàn bộ các đối tượng cây.

In [12]:
def make_mirror_copy(node):
    if node is None:
        return None
    return TreeNode(node.val,
                    make_mirror_copy(node.right),
                    make_mirror_copy(node.left))

asym = tree([1, 2, 2, None, 3, None, 3])
print("Cây gốc  :", show_tree(asym))
print("Bản đảo  :", show_tree(make_mirror_copy(asym)))
print("Bản đảo có phải chính cây ban đầu?", show_tree(asym) == show_tree(make_mirror_copy(asym)))

Cây gốc  : [1, 2, 2, None, 3, None, 3]
Bản đảo  : [1, 2, 2, 3, None, 3]
Bản đảo có phải chính cây ban đầu? False


### 4.2. Nhận ra quy tắc đối chiếu đúng từ một cặp cây con

Ta gọi `mirror(a,b)` là “cây con `a` có phải ảnh gương của cây con `b`?”.
Khi cả hai nút có mặt, so giá trị gốc rồi **bắt chéo**:

- `mirror(a.left, b.right)` — trái ngoài ↔ phải ngoài.
- `mirror(a.right, b.left)` — trái trong ↔ phải trong.

So `a.left` với `b.left` là lỗi thường gặp: nó kiểm tra “hai bên giống nhau” chứ không phải “soi gương”.
Khi vẽ một cặp cạnh, hãy kéo dây nối **chéo qua trục**; đó là cách tự nhớ quy tắc.

In [13]:
left_subtree = TreeNode(2, TreeNode(3), TreeNode(4))
right_subtree = TreeNode(2, TreeNode(4), TreeNode(3))
print("Cặp nút ngoài:", left_subtree.left.val, right_subtree.right.val)
print("Cặp nút trong:", left_subtree.right.val, right_subtree.left.val)
print("Nếu so trái-trái sẽ thấy:", left_subtree.left.val, right_subtree.left.val)

Cặp nút ngoài: 3 3
Cặp nút trong: 4 4
Nếu so trái-trái sẽ thấy: 3 4


## 5. Vì sao cần chuyển sang kỹ thuật chính?

Ta không cần tạo cây gương. So sánh **cặp node phản chiếu**: `(left.left,right.right)` và `(left.right,right.left)`.

Một thuật toán tốt cần trả lời: **ta đang nhớ thông tin gì, cập nhật khi nào, và làm sao không bỏ sót trường hợp nào?**

## 6. Lời giải chính — hiểu từng bước rồi mới viết

Hàm `mirror(a,b)` kiểm tra hai cây có là ảnh gương: hai None hợp lệ; một None sai; giá trị khác sai; còn lại so chéo hai cặp con.

Dưới đây là **hàm giải độc lập**. Khi submit, ta đưa cùng logic vào đúng chữ ký LeetCode.

In [14]:
def solve(root: Optional[TreeNode]) -> bool:
    def mirror(a,b):
        if a is None and b is None: return True
        if a is None or b is None: return False
        return (a.val==b.val and
                mirror(a.left,b.right) and
                mirror(a.right,b.left))
    return root is None or mirror(root.left,root.right)

### 6.1. Bốn quyết định đủ để viết `mirror(a,b)`

1. **Cả hai vị trí trống:** giống ảnh gương ⇒ `True`.
2. **Chỉ một vị trí trống:** không đối xứng ⇒ `False`.
3. **Hai nút có mặt nhưng giá trị khác:** không đối xứng ⇒ `False`.
4. **Giá trị giống:** hai cặp con **bắt chéo** đều phải đối xứng.

Với toàn bộ cây: nếu `root is None`, cây rỗng có thể coi là đối xứng (đề LeetCode thường cho ít nhất một nút,
nhưng kiểm thử mở rộng có thể dùng cây rỗng). Nếu có gốc, ta gọi `mirror(root.left, root.right)`.

**Lỗi kiểu dữ liệu:** `root is None` phải xử lý trước, không thể viết `root.left` khi `root` rỗng.

In [15]:
tiny_symmetric = TreeNode(5, TreeNode(1), TreeNode(1))
tiny_asymmetric = TreeNode(5, TreeNode(1), None)
for sample in [tiny_symmetric, tiny_asymmetric, TreeNode(9), None]:
    print("Cây:", show_tree(sample), "=>", solve(sample))

Cây: [5, 1, 1] => True
Cây: [5, 1] => False
Cây: [9] => True
Cây: [] => True


## 7. Trace bằng tay — quan sát trạng thái, không chỉ đáp án

| Cặp | Vị trí | Kết quả |
|---|---|---|
| `2 ↔ 2` | cặp con gốc | khớp |
| `3 ↔ 3` | trái-trái với phải-phải | khớp |
| `4 ↔ 4` | trái-phải với phải-trái | khớp |
| `None ↔ None` | tới lá | khớp → True |

Đọc từ trái sang phải: mỗi dòng ghi **trạng thái trước khi quyết định**, hành động và phần thay đổi **sau khi quyết định**.

In [16]:
def trace_mirror(a,b,depth=0):
    print(' '*depth,'so ảnh gương:',None if a is None else a.val,None if b is None else b.val)
    if a is None or b is None: return a is b
    if a.val!=b.val: return False
    return trace_mirror(a.left,b.right,depth+2) and trace_mirror(a.right,b.left,depth+2)
r=tree([1,2,2,3,4,4,3]); print('Kết quả:',trace_mirror(r.left,r.right))

 so ảnh gương: 2 2
   so ảnh gương: 3 3
     so ảnh gương: None None
     so ảnh gương: None None
   so ảnh gương: 4 4
     so ảnh gương: None None
     so ảnh gương: None None
Kết quả: True


### 7.1. Trace bằng tay: chọn ví dụ có cấu trúc lệch

Xét `root=[1,2,2,None,3,None,3]`:

```text
       1
      / \
     2   2
      \   \
       3   3
```

Hai nút `3` cùng nằm bên **phải**, không phải soi gương. Trace:

| Bước | Cặp `mirror(a,b)` | Quy tắc | Kết quả |
|---:|---|---|---|
| 1 | `(2,2)` | Gốc khớp, kiểm tra ngoài trước | Chờ |
| 2 | `(None,3)` | Chỉ một bên rỗng | `False` |
| 3 | Trở lại `(2,2)` | `and` đoản mạch: không kiểm tra cặp trong | `False` |
| 4 | Trở về `isSymmetric` | Toàn cây không đối xứng | **`False`** |

**Chỗ tinh tế:** nhiều người thấy cả hai bên đều có số `3` rồi trả `True`, nhưng đã bỏ qua vị trí trái/phải.
Đọc trace bằng các **cặp vị trí**, không chỉ nhìn con số.

### 7.2. Viết phiên bản không đệ quy bằng `deque` (hàng đợi)

`deque` từ `collections` cho phép `.append(x)` để thêm cuối và `.popleft()` để lấy ở đầu
(FIFO: *First In, First Out*). Ta luôn đưa **cặp ảnh gương** vào queue.

Quy tắc không đổi, chỉ thay cách lưu lời gọi chờ:
- Lấy `(a,b)` ra.
- Nếu cả hai `None`: bỏ qua cặp.
- Nếu chỉ một rỗng hoặc giá trị khác: trả `False`.
- Nếu khớp: cho vào queue `(a.left,b.right)` và `(a.right,b.left)`.

Trong bài này hàng đợi cũng có thể chứa `None` như một thành viên của **cặp**, để không bỏ quên vị trí trống.

In [17]:
def is_symmetric_queue(root):
    if root is None:
        return True
    q = deque([(root.left, root.right)])
    while q:
        a, b = q.popleft()
        if a is None and b is None:
            continue
        if a is None or b is None:
            return False
        if a.val != b.val:
            return False
        q.append((a.left, b.right))
        q.append((a.right, b.left))
    return True

for data in [[1,2,2,3,4,4,3], [1,2,2,None,3,None,3], [], [7]]:
    print(data, '->', is_symmetric_queue(tree(data)))

[1, 2, 2, 3, 4, 4, 3] -> True
[1, 2, 2, None, 3, None, 3] -> False
[] -> True
[7] -> True


### 7.3. Trace queue cho từng cặp phản chiếu

Trace in `queue` **trước khi lấy ra** và cặp vừa lấy, rồi ghi lý do quyết định.
Khác với việc chỉ in `True/False`, cách này cho phép bạn tự tìm chỗ thiếu `None`, thiếu cặp hoặc so nhầm hướng.
Cây đối xứng dùng để quan sát đủ các cặp; cây không đối xứng để quan sát nhánh dừng sớm.

In [18]:
def trace_symmetric_queue(root):
    if root is None:
        print("Cây rỗng -> True")
        return True
    q = deque([(root.left, root.right)])
    label = lambda n: None if n is None else n.val
    i = 0
    while q:
        i += 1
        print(f"Bước {i}: queue trước =", [(label(a),label(b)) for a,b in q])
        a, b = q.popleft()
        print(f"  lấy cặp ({label(a)}, {label(b)})")
        if a is None and b is None:
            print("  cả hai rỗng -> tiếp")
            continue
        if a is None or b is None or a.val != b.val:
            print("  KHÁC -> False")
            return False
        q.extend([(a.left, b.right), (a.right, b.left)])
    print("Hết cặp cần so -> True")
    return True

assert trace_symmetric_queue(tree([1,2,2,3,4,4,3]))
assert not trace_symmetric_queue(tree([1,2,2,None,3,None,3]))

Bước 1: queue trước = [(2, 2)]
  lấy cặp (2, 2)
Bước 2: queue trước = [(3, 3), (4, 4)]
  lấy cặp (3, 3)
Bước 3: queue trước = [(4, 4), (None, None), (None, None)]
  lấy cặp (4, 4)
Bước 4: queue trước = [(None, None), (None, None), (None, None), (None, None)]
  lấy cặp (None, None)
  cả hai rỗng -> tiếp
Bước 5: queue trước = [(None, None), (None, None), (None, None)]
  lấy cặp (None, None)
  cả hai rỗng -> tiếp
Bước 6: queue trước = [(None, None), (None, None)]
  lấy cặp (None, None)
  cả hai rỗng -> tiếp
Bước 7: queue trước = [(None, None)]
  lấy cặp (None, None)
  cả hai rỗng -> tiếp
Hết cặp cần so -> True
Bước 1: queue trước = [(2, 2)]
  lấy cặp (2, 2)
Bước 2: queue trước = [(None, 3), (3, None)]
  lấy cặp (None, 3)
  KHÁC -> False


## 8. Vì sao thuật toán đúng? — lập luận bằng bất biến

Hai cây là ảnh gương khi gốc bằng nhau, cây con trái của cây thứ nhất là ảnh gương của cây con phải của cây thứ hai và ngược lại. Base case `None` đúng theo định nghĩa. Quy nạp theo chiều cao bảo đảm kiểm tra đủ mọi cặp node đối xứng.

**Hai câu cần trả lời:** (1) Sau mỗi bước, điều gì vẫn đúng? (2) Khi dừng, vì sao giá trị trả về chính là đáp án theo yêu cầu?

## 9. Code nộp LeetCode (Python 3)

**Chỉ lấy cell bên dưới để nộp.** Phần dựng input và `print()` ở các cell khác chỉ phục vụ học tập.

In [19]:
from typing import Optional
class Solution:
    def isSymmetric(self, root: Optional[TreeNode]):
        def mirror(a,b):
            if a is None and b is None: return True
            if a is None or b is None: return False
            return (a.val==b.val and
                    mirror(a.left,b.right) and
                    mirror(a.right,b.left))
        return root is None or mirror(root.left,root.right)

## 10. Kiểm thử đúng đầu ra, trường hợp biên và lỗi dễ bỏ sót

Các lệnh `assert` sẽ dừng với thông báo lỗi nếu kết quả **khác** giá trị mong đợi. Tập test bao gồm ví dụ dễ hiểu, input nhỏ nhất và tình huống dễ nhầm. Đừng chỉ kiểm tra một ví dụ.

In [20]:
for data,expected in [([1,2,2,3,4,4,3],True),([1,2,2,None,3,None,3],False),([],True),([4],True),([1,2,2,2,None,2],False)]:
    assert solve(tree(data))==expected
    assert Solution().isSymmetric(tree(data))==expected
print('PASS 5 ca đối xứng')

PASS 5 ca đối xứng


### 10.1. Ma trận test nhiều hình cây, đối chiếu baseline và hai cách trực tiếp

Ta cần phân biệt **cùng mức**, **cùng giá trị**, **cùng số nút** với **đối xứng theo trục**.
Các ví dụ dễ nhầm: cây một nút; cây chỉ có con trái; một bên `None`; hai cây con có các giá trị đúng nhưng vị trí thiếu;
tất cả nút có cùng giá trị nhưng hình không cân. Đừng chỉ thử ví dụ đầu tiên của đề.

In [21]:
symmetric_cases = [
    [], [1], [1,2,2], [1,2,2,3,4,4,3],
    [1,2,2,None,3,3,None], [1,2,2,None,3,None,3],
    [8,4,4,None,5,5,None], [8,4,4,5,None,5,None],
    [1,2,2,2,None,2], [0,0,0,0,0,0,0],
]
for vals in symmetric_cases:
    root = tree(vals)
    expected = baseline(root)
    assert solve(root) == expected, vals
    assert is_symmetric_queue(root) == expected, vals
    assert Solution().isSymmetric(root) == expected, vals
print(f"PASS {len(symmetric_cases)} hình cây với 4 cách")

PASS 10 hình cây với 4 cách


### 10.2. Thuộc tính đặc biệt: đảo toàn bộ cây không đổi tính đối xứng

Nếu `T` đối xứng, `mirror_copy(T)` cũng đối xứng; nếu `T` không đối xứng, bản đảo toàn bộ của nó vẫn không đối xứng.
Ta kiểm bằng nhiều cây mẫu, **không cần đoán đáp án** thủ công cho từng hình.
Đây là kiểm thử theo *property* (tính chất): dùng quy luật toán học để sinh cặp test đối chiếu.

In [22]:
for vals in symmetric_cases:
    original = tree(vals)
    flipped = make_mirror_copy(original)
    assert solve(original) == solve(flipped)
    assert is_symmetric_queue(original) == is_symmetric_queue(flipped)
print("PASS tính chất: đối xứng không đổi sau khi phản chiếu toàn cây")

PASS tính chất: đối xứng không đổi sau khi phản chiếu toàn cây


## 11. Phân tích Time/Space Complexity

`n` = số node. **Time O(n)**; **Space O(h)** stack, tệ nhất O(n) khi cây lệch. Baseline còn dựng thêm O(n) node.

Nhớ phân biệt **số phần tử đầu vào** với **độ cao cây**, **số bit** hoặc **số chữ số**. Nếu thuật toán dùng recursion, bộ nhớ ngăn xếp lời gọi cũng cần được tính.

## 12. Các lỗi người mới hay mắc

- ❌ `mirror(a.left,b.left)`: phải bắt cặp chéo.
- ❌ So sánh mỗi giá trị tầng, bỏ qua vị trí `None`.
- ❌ Quên cây rỗng đối xứng.

Khi debug: cố tạo input nhỏ nhất gây sai, trace biến ở **ngay trước lệnh `if`/`return`**, rồi sửa nguyên nhân thay vì thêm điều kiện tùy tiện.

### 12.1. Code sai điển hình: so hai bên **cùng hướng**

Đoạn sai dưới đây trông rất giống lời giải, nhưng `equal(left.left, right.left)` so **trái với trái**.
Nó sẽ bác bỏ một cây thực sự đối xứng `[1,2,2,3,4,4,3]` vì lấy `3` so với `4`.

Muốn soi gương, hãy thay bằng **trái-phải** và **phải-trái**, và vẫn phải giữ điều kiện cả hai `None`.
Ta chạy cả hai kết quả để thấy lỗi, không để cell bị exception.

In [23]:
def wrong_symmetric_same_sides(root):
    def equal(a,b):
        if a is None or b is None:
            return a is b
        return a.val == b.val and equal(a.left,b.left) and equal(a.right,b.right)
    return root is None or equal(root.left,root.right)

sample = tree([1,2,2,3,4,4,3])
print("Code sai:", wrong_symmetric_same_sides(sample))
print("Code đúng:", solve(sample))
assert not wrong_symmetric_same_sides(sample) and solve(sample)

Code sai: False
Code đúng: True


## 13. Nhận diện dạng bài và vận dụng

Nhìn thấy từ **mirror / symmetric / flip** trong cây: chọn hai node rồi đối chiếu **đường đi phản chiếu**, khác với so sánh cây giống nhau.

**Chuyển giao:** hãy tìm một bài khác có cùng dấu hiệu đầu vào và tập trung vào lý do giữ bất biến, không học thuộc tên biến.

## 14. Biến thể có ý nghĩa

Thay `4` phía phải thành `9`: cây vẫn có cùng số node nhưng **không còn đối xứng**.

Biến thể này dùng lại kỹ thuật vừa học, nhưng có một thay đổi nhỏ giúp kiểm tra bạn đã hiểu hay chưa.

In [24]:
print(solve(tree([1,2,2,3,4,9,3])))

False


### 14.1. Phân biệt “Same Tree” (#100) và “Symmetric Tree” (#101)

| Tiêu chí | #100 Same Tree | #101 Symmetric Tree |
|---|---|---|
| Có mấy cây đầu vào? | Hai cây `p, q` | Một cây `root` |
| Hướng ghép con | `a.left` ↔ `b.left` | `a.left` ↔ `b.right` |
| Hướng thứ hai | `a.right` ↔ `b.right` | `a.right` ↔ `b.left` |
| Điều kiện tại nút | Giá trị và sự tồn tại khớp | Giá trị và sự tồn tại khớp |
| Kỹ thuật | Đồng đệ quy hai cây | Đồng đệ quy **chéo** hai nhánh |

Đề mới có từ khóa *mirror, reflection, symmetry* thì hãy vẽ trục gương trước khi ghép con.
Tránh viết theo ký ức của #100: chỉ cần nhầm hướng là hỏng toàn bài.

## 15. Tự luyện — 5 bài tăng dần

**Mục tiêu:** tự hiểu cú pháp, trace bằng tay, viết lại thuật toán, chuyển sang biến thể, rồi thực hiện mini challenge. Tất cả ví dụ giữ nguyên đúng bản chất đề LeetCode. **Đừng xem đáp án trước.**

### Bài 1 — Dự đoán output của Python (dễ)

Với `r = tree([1,2,2,3,4,4,3])`, đoán hai lệnh `print(r.left.left.val, r.right.right.val)` và `print(r.left.right.val, r.right.left.val)`.

Hãy viết output ra giấy trước khi chạy cell dưới.

In [25]:
r=tree([1,2,2,3,4,4,3])
print(r.left.left.val,r.right.right.val)
print(r.left.right.val,r.right.left.val)

3 3
4 4


### Bài 2 — Trace bằng tay (hiểu bản chất)

Trace cây `[1,2,2,None,3,None,3]`: cặp `(left.left,right.right)` gồm những node nào? Cặp `(left.right,right.left)` có gì sai?

Tự viết bảng gồm **trạng thái trước**, phép kiểm tra và **trạng thái sau**; không chỉ ghi kết quả cuối.

### Bài 3 — TODO: Tự viết lời giải chính

Hoàn thiện `exercise_symmetric(root)` bằng hàm phụ `mirror(left,right)` để kiểm tra phản chiếu.

Các hàm bài tập được định nghĩa độc lập và **không được gọi** khi chưa hoàn thiện.

In [26]:
def exercise_symmetric(root):
    # TODO: viết mirror(a,b) với base case None.
    # TODO: so chéo a.left với b.right, a.right với b.left.
    raise NotImplementedError

### Bài 4 — TODO: Chuyển giao sang biến thể

Tách kỹ thuật thành `exercise_mirror_subtrees(a,b)`: hai cây riêng biệt có phải ảnh gương của nhau? Ví dụ `[2,3,4]` và `[2,4,3]` phải trả `True`.

Hãy thử input nhỏ nhất và một phản ví dụ. Dự đoán output trước khi gọi.

In [27]:
def exercise_mirror_subtrees(a, b):
    # TODO: so hai gốc và hai nhánh đối diện, không cần root chung.
    raise NotImplementedError

### Bài 5 — TODO: Mini challenge

Mini challenge: viết `exercise_symmetric_stack(root)` không dùng recursion. Stack chứa cặp node cần soi gương.

Hãy nêu độ phức tạp của hàm và thử tối thiểu 3 ca biên.

In [28]:
def exercise_symmetric_stack(root):
    # TODO: xử lý None, giá trị không khớp, push cặp chéo.
    raise NotImplementedError

### Bài 6 — Đếm số cặp nút không rỗng ở đúng vị trí ảnh gương

**Đề:** `exercise_mirror_pairs(root)` đếm số **cặp hai nút** mà `a` và `b` cùng hiện diện khi soi gương
từ `root.left, root.right`, bất kể giá trị có giống nhau hay không.
Nếu một bên `None`, nhánh đó dừng. Ví dụ `[1,2,2,3,4,4,3]` có 3 cặp ngoài gốc.

**Gợi ý 1:** hàm phụ `count(a,b)` trả `0` nếu một bên rỗng.

<details><summary>Gợi ý 2 — cú pháp</summary>Với hai nút hiện diện: tính <code>1 + ... + ...</code>.</details>

<details><summary>Gợi ý 3 — gần code</summary><code>count(a.left,b.right) + count(a.right,b.left)</code>. Gốc chỉ đóng vai trò chia hai nửa, không tính cặp.</details>

In [29]:
def exercise_mirror_pairs(root):
    # TODO: Đếm các cặp (node_trái, node_phải) đối diện.
    # Không tính chính node root trong tổng.
    raise NotImplementedError

# Sau khi làm: assert exercise_mirror_pairs(tree([1,2,2,3,4,4,3])) == 3

## 16. Gợi ý mở dần — đừng đọc cùng lúc

<details><summary><b>Cấp 1 · Trực giác</b></summary>

Đối xứng không phải trái giống trái; nó là **trái so với phải theo gương**.

</details>

<details><summary><b>Cấp 2 · Dữ liệu và cú pháp</b></summary>

Đệ quy / stack đều lưu cặp `a,b`; cặp kế là `(a.left,b.right)` và `(a.right,b.left)`.

</details>

<details><summary><b>Cấp 3 · Gần-code</b></summary>

Nếu cả hai rỗng thì True; một rỗng thì False; khác giá trị thì False; nếu không, xét cả hai cặp con chéo.

</details>

## 17. Đáp án tham khảo — chỉ đọc sau khi tự làm

### Bài 1 — Output

Hai dòng lần lượt là `3 3` và `4 4`: đối xứng nghĩa là so **chéo** (ngoài với ngoài, trong với trong).

### Bài 2 — Trace tay

Hai gốc con `(2,2)` khớp. Cặp ngoài `(None,3)` khác cấu trúc nên **False** (cũng có thể xét cặp trong `(3,None)` và thấy sai).

### Đáp án bài 3 — Viết lại giải pháp chính

Đọc rõ base case và ý nghĩa giá trị trả về. Đáp án độc lập, không dựa vào `solve` phía trên.

In [30]:
def answer_symmetric(root):
    def mirror(a,b):
        if a is None and b is None: return True
        if a is None or b is None or a.val!=b.val: return False
        return mirror(a.left,b.right) and mirror(a.right,b.left)
    return root is None or mirror(root.left,root.right)

### Đáp án bài 4 — Biến thể

So sánh **trạng thái cần mang theo** giữa bài chính và biến thể.

In [31]:
def answer_mirror_subtrees(a,b):
    if a is None and b is None: return True
    if a is None or b is None or a.val!=b.val: return False
    return answer_mirror_subtrees(a.left,b.right) and answer_mirror_subtrees(a.right,b.left)

### Đáp án bài 5 — Mini challenge

Kiểm tra kỹ trường hợp rỗng, đầu/cuối, và các điều kiện dừng.

In [32]:
def answer_symmetric_stack(root):
    if root is None: return True
    stack=[(root.left,root.right)]
    while stack:
        a,b=stack.pop()
        if a is None and b is None: continue
        if a is None or b is None or a.val!=b.val: return False
        stack.append((a.left,b.right))
        stack.append((a.right,b.left))
    return True

### Đáp án bài 6 — đếm cặp ảnh gương

Ở biến thể này, không bắt buộc `a.val == b.val`; mục tiêu là **đếm cặp tồn tại**.
Vẫn giữ hai bước đệ quy chéo vì đó là cấu trúc đối xứng về vị trí.

In [33]:
def answer_mirror_pairs(root):
    def count(a, b):
        if a is None or b is None:
            return 0
        return (1 + count(a.left, b.right) + count(a.right, b.left))
    return 0 if root is None else count(root.left, root.right)

assert answer_mirror_pairs(tree([1,2,2,3,4,4,3])) == 3
assert answer_mirror_pairs(tree([1,2,2])) == 1
assert answer_mirror_pairs(tree([1,2,None])) == 0
print("PASS bài 6")

PASS bài 6


### Kiểm thử 3 lời giải bài tập

Các assert sau **chỉ kiểm tra đáp án mẫu**, không chạy các hàm TODO của bạn. Khi hoàn thiện TODO, bạn hãy tự thay tên hàm trong test để đánh giá.

In [34]:
for arr,expect in [([1,2,2,3,4,4,3],True),([1,2,2,None,3,None,3],False),([],True),([1],True)]:
    assert answer_symmetric(tree(arr))==expect
    assert answer_symmetric_stack(tree(arr))==expect
assert answer_mirror_subtrees(tree([2,3,4]),tree([2,4,3]))
assert not answer_mirror_subtrees(tree([2,3]),tree([2,3]))
print("PASS: đáp án 3 bài coding của phần luyện tập")

PASS: đáp án 3 bài coding của phần luyện tập


### Tự kiểm trước khi rời bài #101

1. Tại sao `[1,2,2,None,3,None,3]` không đối xứng dù hai con `2` và hai cháu `3` bằng nhau?
2. Vì sao trong `mirror(a,b)` lại cần `(a.left,b.right)` trước?
3. Khi có một `None`, có trường hợp nào còn phải gọi tiếp không?
4. Phiên bản dùng queue khác gì với đệ quy: thay đổi **quy luật** hay chỉ thay đổi cách lưu các cặp đang chờ?
5. Time `O(n)` có nghĩa mỗi nút được xử lý tối đa bao nhiêu lần?
6. Nếu đề muốn so hình dáng ảnh gương mà bỏ qua giá trị, sửa điều kiện nào?

**Luyện nhớ:** tự vẽ hai cây 4 nút, một cây đối xứng, một cây không đối xứng; trace toàn bộ các cặp `a,b` trước khi viết lại `class Solution`.

## 18. Tóm tắt và checklist submit

**Nhớ nhanh:** Ta không cần tạo cây gương. So sánh **cặp node phản chiếu**: `(left.left,right.right)` và `(left.right,right.left)`.

- [ ] Tôi tự nói được input, output và điều kiện biên.
- [ ] Tôi hiểu từng cú pháp Python trong thuật toán.
- [ ] Tôi có thể tự vẽ bảng trace ít nhất một ví dụ.
- [ ] Tôi giải thích được **tại sao đúng** mà không dựa vào test.
- [ ] Tôi phân biệt được Time/Space của baseline và lời giải chính.
- [ ] Tôi copy đúng phần class LeetCode, không copy hàm dựng cây/list hay `print()`.
- [ ] Tôi tự làm xong các TODO, rồi mới đọc đáp án.

**Bài tiếp theo:** duy trì thói quen trace từ trạng thái nhỏ và chỉ tối ưu khi đã hiểu bản chất.